<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-03-prompting/lesson-3.2-structured-output/notebooks/GCP_Capstone_3.2_StructuredOutput.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.2 Structured Output & JSON Mode
**Netsetos GenAI Engineering — GCP Capstone**

Pydantic schemas, response_schema, enums, few-shot examples, anyOf / $ref, and a reusable `structured_output.py` module.

## Setup

In [ ]:
!pip install -q google-genai pydantic
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS

from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import Literal, Optional, Union, List
import json

client = genai.Client(enterprise=True, project=PROJECT_ID, location='us-central1')

## Cell 1: Why Structured Output — Plain Text vs JSON

In [ ]:
# Unstructured: parsing this is a regex nightmare
plain = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Extract: John Doe, 32, john@ex.com, Senior Eng at Acme, $150k',
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0)),
)
print('PLAIN TEXT:')
print(plain.text[:200])

# Structured: parseable, typed, validated
class Person(BaseModel):
    name: str
    age: int
    email: str
    title: str
    salary_usd: int

structured = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Extract: John Doe, 32, john@ex.com, Senior Eng at Acme, $150k',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=Person,
        temperature=0.0,
        thinking_config=types.ThinkingConfig(thinking_budget=0)),
)
p: Person = structured.parsed
print(f'\nSTRUCTURED: name={p.name} age={p.age} salary=${p.salary_usd}')

## Cell 2: Pydantic Field Descriptions Guide the Model

In [ ]:
class DocMetadata(BaseModel):
    title: str = Field(description='Document title, max 120 chars')
    summary: str = Field(description='2-sentence abstract; no marketing language')
    topics: List[str] = Field(description='3-5 distinct topics, lowercase, no duplicates')
    language: Literal['en', 'hi', 'mixed'] = Field(description='Primary language')
    has_pii: bool = Field(description='True if document contains names, emails, phone, Aadhaar, PAN')
    confidence: Literal['high', 'medium', 'low']

text = '''Q4 2025 earnings: Revenue $12.3M (+18% YoY). CEO Priya Sharma noted strong India growth.
Contact: investor@acme.in, +91-98765-43210. PAN: ABCDE1234F.'''

r = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=f'Extract document metadata:\n{text}',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=DocMetadata,
        temperature=0.1,
        thinking_config=types.ThinkingConfig(thinking_budget=0)),
)
meta: DocMetadata = r.parsed
print(json.dumps(meta.model_dump(), indent=2, ensure_ascii=False))

## Cell 3: Enum-Only Output (Classification)

In [ ]:
# Pattern: response_mime_type='text/x.enum' for single-label classification.
# Cheaper + faster than JSON when you only need one label.

SENTIMENT_SCHEMA = {'type': 'STRING', 'enum': ['POSITIVE', 'NEGATIVE', 'NEUTRAL']}

def classify_sentiment(text):
    r = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=f'Sentiment of this review: {text}',
        config=types.GenerateContentConfig(
            response_mime_type='text/x.enum',
            response_schema=SENTIMENT_SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_budget=0)),
    )
    return r.text

for review in [
    'Best onboarding I have ever seen at a company.',
    'Wasted three hours debugging their broken SDK.',
    'The product exists. Nothing more to say.',
]:
    print(f'  {classify_sentiment(review):<8} | {review}')

## Cell 4: Few-Shot Examples in the Prompt (Format Calibration)

In [ ]:
# When the schema is ambiguous, examples beat longer descriptions.

class Invoice(BaseModel):
    vendor: str
    invoice_number: str
    total_inr: float
    due_date: str = Field(description='ISO-8601 YYYY-MM-DD')

FEW_SHOT_PROMPT = '''Extract invoice fields. Follow the format shown.

Example:
Input: Bill from Cloudify Tech. Ref INV-2025-891. Rs. 48,500 due by 15/01/2026.
Output: {"vendor": "Cloudify Tech", "invoice_number": "INV-2025-891", "total_inr": 48500.0, "due_date": "2026-01-15"}

Example:
Input: Acme Corp INV#442 Rs 12,300 pay by 3rd Feb 2026
Output: {"vendor": "Acme Corp", "invoice_number": "INV#442", "total_inr": 12300.0, "due_date": "2026-02-03"}

Now extract:
Input: {input}
Output:'''

inp = 'Invoice from BlueOcean Systems  Ref #BOS/2026/017  Rs 95,750/- payable 31 Mar 2026'
r = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=FEW_SHOT_PROMPT.replace('{input}', inp),
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=Invoice,
        temperature=0.0,
        thinking_config=types.ThinkingConfig(thinking_budget=0)),
)
inv: Invoice = r.parsed
print(f'{inv.vendor} | {inv.invoice_number} | Rs {inv.total_inr:,.0f} | due {inv.due_date}')

## Cell 5: Nested Schemas & Lists

In [ ]:
class Citation(BaseModel):
    chunk_id: int
    quote: str = Field(description='Exact quote from source, max 200 chars')

class RAGAnswer(BaseModel):
    answer: str = Field(description='Answer grounded in the provided context')
    citations: List[Citation] = Field(description='All chunks that support the answer')
    confidence: Literal['high', 'medium', 'low']
    answerable: bool = Field(description='False if context does not contain the answer')

CONTEXT = '''[1] DocuMind supports PDF, DOCX, TXT, and Markdown files up to 200 MB.
[2] Uploads are processed via Doc AI Layout Parser v1.5 with 500-token chunks.
[3] Embeddings use text-embedding-005 at 768 dimensions.'''

r = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=f'Answer only from the context.\n\nContext:\n{CONTEXT}\n\nQuestion: Which file types are supported and what is the size limit?',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=RAGAnswer,
        temperature=0.0,
        thinking_config=types.ThinkingConfig(thinking_budget=0)),
)
ans: RAGAnswer = r.parsed
print(f'Answerable: {ans.answerable} | Confidence: {ans.confidence}')
print(f'Answer: {ans.answer}')
for c in ans.citations:
    print(f'  [{c.chunk_id}] "{c.quote}"')

## Cell 6: Discriminated Union (anyOf) — Multi-Intent Routing

In [ ]:
# Discriminated unions let the model pick a branch based on intent.
# The Gemini API materialises Pydantic Union[...] as JSON Schema anyOf.

class SearchIntent(BaseModel):
    action: Literal['search']
    query: str
    top_k: int = 10

class SummarizeIntent(BaseModel):
    action: Literal['summarize']
    document_id: str
    length: Literal['short', 'medium', 'long']

class TranslateIntent(BaseModel):
    action: Literal['translate']
    text: str
    target_language: Literal['en', 'hi', 'ta', 'bn', 'te']

class Intent(BaseModel):
    intent: Union[SearchIntent, SummarizeIntent, TranslateIntent] = Field(discriminator='action')
    confidence: float = Field(ge=0.0, le=1.0)

utterances = [
    'Find me the last five deployment post-mortems.',
    'Give me a long summary of document doc-9f23.',
    'Translate this to Hindi: The deadline is next Monday.',
]
for u in utterances:
    r = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=f'Classify user intent: {u}',
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=Intent,
            temperature=0.0,
            thinking_config=types.ThinkingConfig(thinking_budget=0)),
    )
    i: Intent = r.parsed
    print(f'{i.intent.action:<10} conf={i.confidence:.2f}  |  {u}')

## Cell 7: When Schema Fails — Guardrails & Fallback Parsing

In [ ]:
# Even with response_schema the parsed value can be None on stop-token truncation.
# Always guard with .parsed is None and fall back to raw JSON parsing.

def safe_extract(client, prompt, schema):
    r = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=schema,
            temperature=0.0,
            max_output_tokens=2048,
            thinking_config=types.ThinkingConfig(thinking_budget=0)),
    )
    if r.parsed is not None:
        return r.parsed, 'parsed'
    try:
        return schema.model_validate_json(r.text), 'fallback-json'
    except Exception:
        return None, f'failed: {r.text[:80]}'

class SimpleFact(BaseModel):
    fact: str
    source: Optional[str] = None

val, status = safe_extract(client, 'What year did the Apollo 11 mission land?', SimpleFact)
print(f'status={status} | value={val}')

## Cell 8: structured_output.py — Reusable Module

In [ ]:
# DocuMind structured_output module: the six canonical schemas reused across Modules 4-12.

class RAGAnswer(BaseModel):
    answer: str
    citations: List[int]
    confidence: Literal['high','medium','low']
    answerable: bool

class DocMetadata(BaseModel):
    title: str
    summary: str
    topics: List[str]
    language: Literal['en','hi','mixed']
    has_pii: bool

class ExtractedEntity(BaseModel):
    kind: Literal['person','org','location','date','money','pii']
    text: str
    start: int
    end: int

class ClassificationResult(BaseModel):
    label: str
    confidence: float = Field(ge=0, le=1)
    reasoning: Optional[str] = None

class EvalJudgement(BaseModel):
    relevance: int = Field(ge=1, le=5)
    faithfulness: int = Field(ge=1, le=5)
    helpfulness: int = Field(ge=1, le=5)
    rationale: str

SCHEMAS = {
    'rag': RAGAnswer,
    'doc_meta': DocMetadata,
    'entities': List[ExtractedEntity],
    'classify': ClassificationResult,
    'judge': EvalJudgement,
    'intent': Intent,
}

def structured(prompt, schema_key, model='gemini-3.6-flash', temp=0.0):
    schema = SCHEMAS[schema_key]
    r = client.models.generate_content(
        model=model, contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=schema, temperature=temp,
            thinking_config=types.ThinkingConfig(thinking_budget=0)),
    )
    return r.parsed

# Quick test
j = structured('Rate this answer "Python is snake." for the question "What is Python?"', 'judge')
print(f'Judge: rel={j.relevance} faith={j.faithfulness} help={j.helpfulness}')
print(f'Reason: {j.rationale}')
print('\nModule ready: structured(prompt, schema_key, model, temp)')

## ✅ Lesson 3.2 Complete!

- ✅ Pydantic `BaseModel` + `response_schema` for typed extraction
- ✅ `Field(description=...)` to guide the model without bloating the prompt
- ✅ `response_mime_type='text/x.enum'` for cheap single-label classification
- ✅ Few-shot examples in the prompt when schema alone is ambiguous
- ✅ Nested schemas (`List[Citation]`) for RAG answers with sources
- ✅ Discriminated `Union[...]` (anyOf) for multi-intent routing
- ✅ `.parsed is None` guard + fallback `model_validate_json` for truncation
- ✅ `structured_output.py` module with six canonical schemas (RAG, metadata, entities, classify, judge, intent)

**Next: Lesson 3.3 — Chain-of-Thought & Model Routing**